# Single magnet comparison between the two algorithms

This notebook is an example of the single magnet simulation within Figure 2 used to compare the Levenberg–Marquardt (LM) implementation with the INFO filter. Both algorithms receive the same initial guess for each attempt and it compares the estimated positions. 

The default quick preset uses two depths and two positions to confirm your environment is set up properly. Set `FULL_BENCHMARK = True` in the configuration cell to reproduce the paper sweep: 13 positions, depths 30/40/50/60 mm, two trials and initial guesses per position, 4,000 iterations, and the 500 iteration window for the accuracy and precision evaluation. 

## Requirements

Please install the required packages and libraries in 'requirements.txt'!


In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_candidates = (Path.cwd(), Path.cwd().parent)
repo_root = next(
    (candidate for candidate in repo_candidates if (candidate / "Magnet-Tracking.py").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Please ensure the .py file is in your path / directory")
module_path = repo_root / "Magnet-Tracking.py"

spec = spec_from_file_location("magnet_tracking", module_path)
magnet_tracking = module_from_spec(spec)
spec.loader.exec_module(magnet_tracking)
MagnetTrackingSystem = magnet_tracking.MagnetTrackingSystem


## Configuration

The sensor layout matches the paper's configuration: a 20 by 16 by 2 grid with 3.6, 2.7, and 3.0 mm spacing respectively, with sensors on both sides (seperated in the z direction).

In [ ]:
FULL_BENCHMARK = False

MAE_THRESHOLD_MM = 10.0 # this is to restart with a new initial condition if either algorithm fails to converge to the non convexity of the dipole field structure.

# INFO filter parameters used for the simulation study. Tau determines the second-order state model dynamics, 
# while noise_std is to tune trust in the measurements, while position and moment are trust for the state model. 
TAU = 0.5
GEO_TAU = 0.5
GEO_STD = 10
NOISE_STD = 0.4
POSITION_STD = 15
MOMENT_STD = 15

def sensor_positions():
    xs = np.arange(0, 41, 10)
    ys = np.arange(0, 33, 8)
    return (
        [(x, ys[0]) for x in xs[1:]]
        + [(xs[0], y) for y in ys[1:]]
        + list(zip(xs, ys)))

if FULL_BENCHMARK:
    ITERS_TO_RUN = 4000
    NUM_TRIALS = 2
    VAR_ITERS = 500
    ACTIVE_DEPTHS = [30, 40, 50, 60]
    POSITIONS_XY = sensor_positions()
    MAX_ATTEMPTS_PER_SYSTEM = 100
else:
    ITERS_TO_RUN = 4000
    NUM_TRIALS = 1
    VAR_ITERS = 500
    ACTIVE_DEPTHS = [30, 50]
    POSITIONS_XY = [(0, 0), (20, 16)]
    MAX_ATTEMPTS_PER_SYSTEM = 8

print(
    f"Preset: {'full benchmark' if FULL_BENCHMARK else 'quick validation'}"
)

## Build the simulated systems

In [ ]:
system_params = dict(
    is_euclid=False,
    num_magnets=1,
    width=20,
    length=16,
    height=2,
    spacing=(3.6, 2.7, 3.0),
    center=(0.0, 0.0, 1.5),
    pos_tau=TAU,
    mom_tau=TAU,
    geo_tau=GEO_TAU,
    dt=1e-3,
    pos_std=POSITION_STD,
    mom_std=MOMENT_STD,
    geo_std=GEO_STD,
    noise_std=NOISE_STD,
)

systems_by_depth = {depth: [] for depth in ACTIVE_DEPTHS}
for depth in ACTIVE_DEPTHS:
    for x_pos, y_pos in POSITIONS_XY:
        system = MagnetTrackingSystem(**system_params)
        true_params = np.array([[x_pos, y_pos, depth, 90.0, 0.0]])
        system.set_with_angles(true_params, ret=False)
        systems_by_depth[depth].append(system)

example = systems_by_depth[ACTIVE_DEPTHS[0]][0]

print(f"Created {sum(map(len, systems_by_depth.values()))} systems with {example.total_num_sensors} sensors each.")

## Positional error metric functions

In [ ]:
def calculate_positional_metrics(run_states, system_obj, iterations_to_analyze):
    """Return final metrics"""
    true_position = system_obj.true_params[0, :3]
    estimated_positions = run_states[-iterations_to_analyze:, :3]
    errors = np.linalg.norm(estimated_positions - true_position, axis=1)
    return {
        "mae": float(np.mean(errors)),
        "variance": float(np.var(errors)),
        "rmse": float(np.sqrt(np.mean(errors**2))),
        "raw": errors,
    }

def calculate_positional_mae(run_states, system_obj, iterations_to_analyze):
    return calculate_positional_metrics(run_states, system_obj, iterations_to_analyze)["mae"]

## Run EIF and LM from initial guesses


In [ ]:
all_steady_EIFs = {depth: [] for depth in ACTIVE_DEPTHS}
all_systems_for_EIF_runs = {depth: [] for depth in ACTIVE_DEPTHS}
all_steady_LMs = {depth: [] for depth in ACTIVE_DEPTHS}
all_systems_for_LM_runs = {depth: [] for depth in ACTIVE_DEPTHS}
all_initial_guesses = {depth: [] for depth in ACTIVE_DEPTHS}
successful_runs = []

eif_params = dict(
    iter=ITERS_TO_RUN, all_states=True, with_geo=True, with_noise=True, verbose=0
)
lm_params = dict(iter=ITERS_TO_RUN, with_geo=True, with_noise=True, verbose=0)

for depth in ACTIVE_DEPTHS:
    print(f"\n--- Processing depth: {depth} mm ---")
    for system_index, system_obj in enumerate(systems_by_depth[depth]):
        xy = tuple(system_obj.true_params[0, :2])
        print(f"  Position {system_index + 1}/{len(POSITIONS_XY)}: {xy}")

        for trial_index in range(NUM_TRIALS):
            accepted = False
            for attempt in range(1, MAX_ATTEMPTS_PER_SYSTEM + 1):
                initial_guess = system_obj.random_initial_guess(with_geo=True)

                eif_states = system_obj.run_EIF(
                    initial_guess=initial_guess.copy(), **eif_params
                )[0]
                lm_states = system_obj.run_LM_loop(
                    initial_guess=initial_guess.copy(), **lm_params
                )[0]

                eif_metrics = calculate_positional_metrics(eif_states, system_obj, VAR_ITERS)
                lm_metrics = calculate_positional_metrics(lm_states, system_obj, VAR_ITERS)
                print(
                    f"    attempt {attempt}: EIF {eif_metrics['mae']:.3f} mm | "
                    f"LM {lm_metrics['mae']:.3f} mm"
                )

                if eif_metrics["mae"] < MAE_THRESHOLD_MM and lm_metrics["mae"] < MAE_THRESHOLD_MM:
                    all_steady_EIFs[depth].append(eif_states)
                    all_systems_for_EIF_runs[depth].append(system_obj)
                    all_steady_LMs[depth].append(lm_states)
                    all_systems_for_LM_runs[depth].append(system_obj)
                    all_initial_guesses[depth].append(initial_guess.copy())
                    successful_runs.append(
                        {
                            "depth": depth,
                            "position": xy,
                            "trial": trial_index + 1,
                            "attempt": attempt,
                            "initial_guess": initial_guess.copy(),
                            "EIF": eif_metrics,
                            "LM": lm_metrics,
                        }
                    )
                    accepted = True
                    print("    accepted")
                    break

            if not accepted:
                raise RuntimeError(
                    f"No matched successful run for depth={depth}, position={xy}, "
                    f"trial={trial_index + 1} after {MAX_ATTEMPTS_PER_SYSTEM} attempts."
                )

print(f"\nCompleted {len(successful_runs)} successful runs.")

## Summary

In [ ]:
print(
    f"{'Depth':>7} "
    f"{'Algorithm':>10} "
    f"{'Runs':>6} "
    f"{'Mean MAE (mm)':>15} "
    f"{'Mean Variance (mm²)':>20}"
)
print("-" * 65)

for depth in ACTIVE_DEPTHS:
    for algorithm in ("EIF", "LM"):
        rows = [row[algorithm] for row in successful_runs if row["depth"] == depth]

        mean_mae = np.mean([row["mae"] for row in rows])

        mean_variance = np.mean([row["variance"]for row in rows])

        print(
            f"{depth:7d} "
            f"{algorithm:>10} "
            f"{len(rows):6d} "
            f"{mean_mae:15.4f} "
            f"{mean_variance:20.6f}")

## Plotting

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# Convert into expected plotting format
all_eif_pos_errors = {
    depth: {
        "mean": [
            row["EIF"]["mae"]
            for row in successful_runs
            if row["depth"] == depth
        ]
    }
    for depth in ACTIVE_DEPTHS
}

all_lm_pos_errors = {
    depth: {
        "mean": [
            row["LM"]["mae"]
            for row in successful_runs
            if row["depth"] == depth
        ]
    }
    for depth in ACTIVE_DEPTHS
}

# plot font type
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.sans-serif"] = ["Arial"]
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

eif_color = "#A77CD9"
lm_color = "#949494"

# plot dimensioning
def mm_to_inches(width_mm, height_mm):
    return width_mm / 25.4, height_mm / 25.4

# plot function
def plot_barplot(
    all_eif_errors,
    all_lm_errors,
    error_key,
    ylabel,
    title_prefix,
    save_name,
    use_log=True,
):
    """Plot EIF versus LM error at each active depth."""

    depth_labels = list(ACTIVE_DEPTHS)

    eif_data = [
        np.asarray(
            all_eif_errors.get(depth, {}).get(error_key, []),
            dtype=float,
        )
        for depth in depth_labels
    ]
    lm_data = [
        np.asarray(
            all_lm_errors.get(depth, {}).get(error_key, []),
            dtype=float,
        )
        for depth in depth_labels
    ]

    for depth, eif_vals, lm_vals in zip(
        depth_labels,
        eif_data,
        lm_data,
    ):
        print(f"\nDepth {depth}mm:")
        print(
            f"  EIF: n={len(eif_vals)}, "
            f"mean={np.mean(eif_vals):.4f}+/-{stats.sem(eif_vals):.4f}"
        )
        print(
            f"  LM:  n={len(lm_vals)}, "
            f"mean={np.mean(lm_vals):.4f}+/-{stats.sem(lm_vals):.4f}"
        )

    # Create figure
    fig, ax = plt.subplots(figsize=mm_to_inches(100, 200))

    bar_width = 0.4
    spacing_factor = 1
    x_pos = np.arange(len(depth_labels)) * spacing_factor
    eif_pos = x_pos - bar_width / 2
    lm_pos = x_pos + bar_width / 2

    eif_means = [
        np.mean(values) if len(values) else 0
        for values in eif_data
    ]
    lm_means = [
        np.mean(values) if len(values) else 0
        for values in lm_data
    ]

    eif_sems = [
        stats.sem(values) if len(values) > 1 else 0
        for values in eif_data
    ]
    lm_sems = [
        stats.sem(values) if len(values) > 1 else 0
        for values in lm_data
    ]

    ax.bar(
        eif_pos,
        eif_means,
        bar_width,
        yerr=eif_sems,
        color=eif_color,
        alpha=0.7,
        capsize=2,
        error_kw={
            "capthick": 2,
            "elinewidth": 2,
        },
        edgecolor="none",
    )

    ax.bar(
        lm_pos,
        lm_means,
        bar_width,
        yerr=lm_sems,
        color=lm_color,
        alpha=0.7,
        capsize=2,
        error_kw={
            "capthick": 2,
            "elinewidth": 2,
        },
        edgecolor="none",
    )

    # Individual trial points
    point_colors = {
        "EIF": "#664785",
        "LM": "#737373",
    }
    jitter_strength = 0.05

    for index, (eif_vals, lm_vals) in enumerate(
        zip(eif_data, lm_data)
    ):
        eif_x = np.random.normal(
            loc=eif_pos[index],
            scale=jitter_strength,
            size=len(eif_vals),
        )
        ax.plot(
            eif_x,
            eif_vals,
            "o",
            color=point_colors["EIF"],
            markersize=5,
            alpha=0.6,
            zorder=3,
            markeredgecolor="white",
            markeredgewidth=0.1,
            linestyle="None",
        )

        lm_x = np.random.normal(
            loc=lm_pos[index],
            scale=jitter_strength,
            size=len(lm_vals),
        )
        ax.plot(
            lm_x,
            lm_vals,
            "o",
            color=point_colors["LM"],
            markersize=5,
            alpha=0.6,
            zorder=3,
            markeredgecolor="white",
            markeredgewidth=0.1,
            linestyle="None",
        )

    if use_log:
        ax.set_yscale("log")

    # Axis formatting
    ax.grid(False)
    ax.set_ylabel(
        ylabel,
        labelpad=1,
        fontsize=12,
    )
    ax.set_xlabel(
        "Depth (mm)",
        labelpad=1,
        fontsize=12,
    )

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=12,
        width=0.4,
    )
    ax.tick_params(
        axis="both",
        which="minor",
        width=0.4,
    )

    ax.set_xticks(x_pos)
    ax.set_xticklabels(depth_labels)

    ax.spines["left"].set_linewidth(0.4)
    ax.spines["bottom"].set_linewidth(0.4)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)

    plt.tight_layout()
    plt.savefig(
        save_name,
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


plot_barplot(
    all_eif_pos_errors,
    all_lm_pos_errors,
    "mean",
    "Mean Position Error (mm)",
    "Mean Positional Error",
    "single_mag_mean_pos_error_comparison.png",
)